# Thinking Spatially: Mapping Cross-Border Trade

**Lesson 5: GeoPandas** — from tables to maps.

**Scenario.** You are a food-security analyst at FEWS NET. You have informal cross-border trade
records for South Africa and you want to put them *on a map*: which neighbours does South Africa
trade with, how much, and along what corridor?

**Data**
- `sa_cross_border_trade.json` — FEWS NET monitored informal trade for South Africa
  ([source](https://fdw.fews.net/api/tradeflowquantityvaluefacts/?dataset=1845&country=ZA&fields=simple&format=json))
- `africa_countries.geojson` — country boundary polygons (Natural Earth)
- `sadc_capitals.csv` — capital-city coordinates for Southern African countries

Remember the big idea from the lesson: **a map is just a scatterplot with rules** — longitude on
X, latitude on Y — and a `GeoDataFrame` is a regular DataFrame with a special `geometry` column.

In [ ]:
pip install geopandas contextily mapclassify

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))
from src.utilities.project_paths import RAW_DIR

GEO_DIR           = RAW_DIR / 'geo'
COUNTRIES_GEOJSON = GEO_DIR / 'africa_countries.geojson'
CAPITALS_CSV      = GEO_DIR / 'sadc_capitals.csv'
TRADE_JSON        = RAW_DIR / 'fews_net' / 'sa_cross_border_trade.json'

# Southern-African countries used to zoom the maps (ISO A2 codes)
SADC = ['ZA', 'ZW', 'MW', 'ZM', 'MZ', 'BW', 'NA', 'LS', 'SZ', 'AO', 'TZ']

pd.set_option('display.float_format', '{:,.1f}'.format)
print('geopandas', gpd.__version__, '| contextily', cx.__version__)

---
# Part A — From a Table to a Map

A `GeoDataFrame` is a regular DataFrame plus a `geometry` column that stores shapes (points,
lines, polygons) and knows how to draw them. In this part we build three layers:
the **trade table**, the **country polygons**, and the **capital points**.

## A1. The trade table

Load the FEWS NET records, convert the common unit (kilograms) to tonnes, and total the volume
by trading **partner** (the country that is *not* South Africa).

In [ ]:
raw = pd.DataFrame(json.load(open(TRADE_JSON)))
print(f'{len(raw):,} records')

# tonnes from the common unit (kilograms); partner = the country that is not South Africa
raw['tonnes']      = pd.to_numeric(raw['common_unit_quantity'], errors='coerce') / 1000
raw['partner']     = raw['destination'].where(raw['source'] == 'South Africa', raw['source'])
raw['partner_iso'] = raw['destination_country_code'].where(raw['source'] == 'South Africa',
                                                           raw['source_country_code'])

# TODO: group by ['partner', 'partner_iso'] -> tonnes = sum, records = count; sort by tonnes desc
trade = ...
display(trade)

## A2. The country polygons (GeoJSON)

Load the boundary polygons with `gpd.read_file()`. Notice the `geometry` column holds `POLYGON`
shapes and the layer carries a **CRS** (coordinate reference system).

In [ ]:
# TODO: load COUNTRIES_GEOJSON with gpd.read_file(...)
countries = ...

print('CRS        :', countries.crs)
print('rows, cols :', countries.shape)
display(countries[['name', 'iso_a2', 'pop_est']].head())

# TODO: plot the boundaries (color='lightgray', edgecolor='white')

## A3. The capital points

The capitals arrive as plain `lat`/`lon` numbers. Turn them into a points `GeoDataFrame`.

⚠️ `Point` takes `(x, y)` — that is **`(longitude, latitude)`**, not `(lat, lon)`.

In [ ]:
caps = pd.read_csv(CAPITALS_CSV)

# TODO: build a GeoDataFrame of capital points
#   geometry = [Point(lon, lat) for lon, lat in zip(caps['lon'], caps['lat'])]
#   remember Point(x, y) = (longitude, latitude)!
#   crs = 'EPSG:4326'
capitals = ...
display(capitals.head())

# TODO: plot the SADC boundaries, then plot the capitals on the same axes (ax=...)

---
# Part B — Spatial Join: Which Country Is Each Point In?

A spatial join matches rows by **where they are**, not by a shared ID. Instead of "do these rows
share a key?" we ask "is this point **within** that polygon?" — the *Spatial VLOOKUP*.

## B1. Tag each capital with its country

Use `gpd.sjoin(..., predicate='within', how='left')` to attach the polygon attributes
(`name`, `pop_est`, ...) to every capital point.

In [ ]:
# TODO: spatial join capitals INTO countries (predicate='within', how='left')
capitals_tagged = ...

print('unmatched:', capitals_tagged['name'].isna().sum())
display(capitals_tagged[['capital', 'country', 'name', 'pop_est']])

## B2. Why `predicate` and coordinate order matter (Pitfall #1)

What if we had swapped longitude and latitude when building the points? The points land in the
wrong place — often in the ocean — and an **inner** join silently drops them.

In [ ]:
# TODO: build 'swapped' points the WRONG way -- Point(lat, lon) instead of Point(lon, lat)
swapped = ...

swap_inner = gpd.sjoin(swapped, countries, how='inner', predicate='within')
swap_left  = gpd.sjoin(swapped, countries, how='left',  predicate='within')
print(f'Swapped order -> inner keeps {len(swap_inner)} of {len(caps)}')
print(f'Swapped order -> left  keeps {len(swap_left)} (unmatched NaN: {swap_left["name"].isna().sum()})')

---
# Part C — Join the Trade onto the Map, and Build Flows

Now attach the trade volumes to the country polygons (a plain attribute merge on the ISO code),
and build **flow lines** from South Africa to each partner.

## C1. Attribute join: trade tonnes onto polygons

Merge `trade` onto `countries` with a **left** join so every country is kept — partners get a
volume, everyone else gets `NaN` (Pitfall #3: don't drop them with an inner join).

In [ ]:
# TODO: left-merge trade[['partner_iso','tonnes']] onto countries
#        left_on='iso_a2', right_on='partner_iso', how='left'
map_data = ...

print('countries with trade data:', map_data['tonnes'].notna().sum())
display(map_data.loc[map_data.tonnes.notna(), ['name', 'iso_a2', 'tonnes']])

## C2. Build flow lines

A flow is a `LineString` from South Africa's capital to a partner's capital. Build one per partner
that has a recorded volume.

In [ ]:
sa_point    = capitals.loc[capitals.iso_a2 == 'ZA', 'geometry'].iloc[0]
partner_pts = capitals.set_index('iso_a2')['geometry']

# TODO: for each partner in trade with tonnes > 0, build a
#       LineString([sa_point, partner_pts[partner_iso]]) and collect into flow_rows
flow_rows = []
for _, r in trade[trade.tonnes > 0].iterrows():
    ...

flows = gpd.GeoDataFrame(flow_rows, crs='EPSG:4326')
display(flows[['partner', 'tonnes']])

---
# Part D — The Multi-Layer Map

Think of a map as a layer cake: **basemap** at the bottom, **boundaries / choropleth** in the
middle, **your data** (flows, points) on top. Plot from top to bottom and control the stacking with
`zorder`. Convert everything to **Web Mercator (EPSG:3857)** first so `contextily` basemaps line up
(Pitfall #2).

## D1. Assemble the layers

In [ ]:
# Project every layer to Web Mercator for contextily
map_web      = map_data.to_crs(3857)
flows_web    = flows.to_crs(3857)
capitals_web = capitals.to_crs(3857)
sadc_web     = countries[countries.iso_a2.isin(SADC)].to_crs(3857)

fig, ax = plt.subplots(figsize=(11, 11))

# TODO: build the layer cake (top layers first, basemap last):
# 1. map_web.plot(column='tonnes', cmap='OrRd', legend=True,
#                 missing_kwds={'color': '#e8e8e8'}, zorder=2)
# 2. outline South Africa: map_web[map_web.iso_a2 == 'ZA'].plot(
#                 facecolor='none', edgecolor='black', linewidth=2, zorder=3)
# 3. draw each flow line with ax.plot(*geom.xy, color='navy', linewidth=..., zorder=4)
# 4. capitals_web.plot(color='black', markersize=25, zorder=5)  + labels
# 5. cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)   # wrap in try/except
# 6. zoom to sadc_web.total_bounds, then ax.set_axis_off()
plt.show()

---
# Part E — Common Pitfalls (Debugging Checklist)

| Symptom | Likely cause | Fix |
|---|---|---|
| Points in the wrong place | lat/lon swapped | use `Point(lon, lat)` |
| Basemap misaligned / tiny dot | wrong projection | `.to_crs(epsg=3857)` before `add_basemap` |
| Rows disappear after `sjoin` | inner join | use `how='left'`, then count `NaN` |
| File won't load | missing shapefile parts | ship the `.zip`/`.gpkg`, not a lone `.shp` |
| Spatial join gives nonsense | CRS mismatch | align with `.to_crs()` before the join |

**When in doubt, plot it** — a quick `.plot()` reveals most problems faster than any print.

### Quick Check ✓

1. What is the difference between a regular `merge()` and a spatial `sjoin()`?
2. Why must you convert to `EPSG:3857` before calling `cx.add_basemap()`?
3. In C1 the left join kept 51 countries but only 2 had trade values. Why keep the other 49?